# 📊 Notebook 2: Model Evaluatie, Error Analyse & IoU / NMS Benchmark
### Sports Vision & Tactical Mapping Challenge | Thomas More Geel
**Team:** Git Push and Pray  
**Auteur:** Dmitrij Tchervonisjenko (met Kobe Vandenberk, Cisse Vandeweyer, Brent Van Gelder)  
**Vak:** Deep Learning & Computer Vision (3e Jaar)  
**Doel:** Diepgaande foutanalyse, wiskundige evaluatiemetrics (PR vs. ROC, mAP ontrafeld) en de systematische IoU/NMS benchmark.

---

## Inhoudsopgave
1. [Evaluatiemetrics & Wiskundige Grondslagen](#1-evaluatiemetrics--wiskundige-grondslagen)
2. [Cruciale Examentheorie: Waarom NOOIT een ROC-Curve bij Objectdetectie?](#2-cruciale-examentheorie-waarom-nooit-een-roc-curve-bij-objectdetectie)
3. [mAP Ontrafeld ("Mean of Averages") [EXAMEN VRAAG]](#3-map-ontrafeld-mean-of-averages-examen-vraag)
4. [IoU (Intersection over Union) & NMS (Non-Maximum Suppression)](#4-iou-intersection-over-union--nms-non-maximum-suppression)
5. [Systematische Benchmark van 8 IoU & NMS Configuraties](#5-systematische-benchmark-van-8-iou--nms-configuraties)
6. [De Winnende Configuratie & Verantwoording](#6-de-winnende-configuratie--verantwoording)
7. [Kwalitatieve Error Analyse: Voetocclusies & Penaltystip Hallucinaties](#7-kwalitatieve-error-analyse-voetocclusies--penaltystip-hallucinaties)

---
## 1. Evaluatiemetrics & Wiskundige Grondslagen

Bij binaire en multi-class evaluatie hanteren we de contingentiematrix:
* **True Positive ($TP$):** Een grond-waarheid object dat correct gedetecteerd is met $\text{IoU} \ge \text{threshold}$.
* **False Positive ($FP$):** Een gedetecteerde bounding box waar géén object staat, of met $\text{IoU} < \text{threshold}$, of een duplicaatdetectie.
* **False Negative ($FN$):** Een grond-waarheid object dat volledig gemist is door het model.
* **True Negative ($TN$):** Een achtergrondlocatie waar het model correct géén bounding box voorspelt.

### Formules:
* **Precisie (Precision - Zuiverheid):**
  $$P = \frac{TP}{TP + FP}$$
  *Hoeveel van onze voorspellingen zijn daadwerkelijk correct?*
* **Recall (Dekking / Gevoeligheid):**
  $$R = \frac{TP}{TP + FN}$$
  *Hoeveel van de aanwezige grond-waarheid objecten hebben we gevonden?*
* **F1-Score (Harmonisch Gemiddelde):**
  $$F_1 = 2 \cdot \frac{P \cdot R}{P + R} = \frac{2TP}{2TP + FP + FN}$$
  Het harmonisch gemiddelde bestraft extreme onevenwichtigheden zwaarder dan het rekenkundig gemiddelde.

---
## 2. Cruciale Examentheorie: Waarom NOOIT een ROC-Curve bij Objectdetectie?

In traditionele machine learning en medische diagnostiek is de **ROC-Curve** (Receiver Operating Characteristic) populair, waarbij de *True Positive Rate (Recall)* wordt uitgezet tegen de *False Positive Rate (FPR)*:

$$\text{TPR} = \frac{TP}{TP + FN}, \quad \text{FPR} = \frac{FP}{FP + TN}$$

### Waarom is de ROC-Curve fundamenteel ONGELDIG voor Objectdetectie?
1. **Het Oneindige Achtergrond Probleem ($TN \to \infty$):**
   In een Full HD broadcast videobeeld van $1920 \times 1080$ pixels zijn er slechts $\approx 23$ objecten aanwezig (22 spelers + 1 bal). 
   Echter, het aantal mogelijke achtergrondvensters van willekeurige schaal en positie waar géén object staat, is letterlijk ontelbaar ($TN > 100.000$ per frame).
2. **De Wiskundige Instorting van FPR:**
   Omdat de noemer $FP + TN$ gedomineerd wordt door $TN$, geldt:
   $$\text{FPR} = \frac{FP}{FP + TN} \approx \frac{FP}{\infty} \approx 0.0000...$$
   Zelfs als het model honderden foutieve hallucinaties genereert ($FP = 500$), blijft de FPR nagenoeg **$0$**!
3. **Het Gevolg:**
   De ROC-curve klampt zich direct vast aan de linkerbovenhoek en toont een bedrieglijk perfecte $\text{AUC-ROC} \approx 0.999$, waardoor ernstige detectiefouten volledig worden gemaskeerd.
4. **De Enige Juiste Metric: De Precision-Recall (PR) Curve:**
   De PR-curve bevat **géén $TN$ in de formule**. Hij evalueert uitsluitend $TP$, $FP$ en $FN$, waardoor hij de zeldzame klasse (zoals een kleine voetbal) eerlijk en genadeloos beoordeelt.

---
## 3. mAP Ontrafeld ("Mean of Averages") [EXAMEN VRAAG]

Een gegarandeerde vraag op het examen Deep Learning: **Wat betekent mAP precies en waarom zitten er twee gemiddelden in de naam?**

* Wiskundig en taalkundig zijn **Mean** en **Average** synoniemen: beide betekenen het *rekenkundig gemiddelde*.
* Toch heet de metric **Mean Average Precision (mAP)** omdat er twee hiërarchische niveaus van middeling plaatsvinden:

### Niveau 1: Average Precision (AP) per Klasse
Voor één specifieke klasse (bijv. 'bal') berekenen we de precisie $P(r)$ als functie van recall $r$ over alle mogelijke betrouwbaarheidsdrempels (confidence thresholds van 1.0 naar 0.0). De **Average Precision (AP)** is de oppervlakte onder deze PR-curve (AUC-PR):

$$\text{AP} = \int_0^1 p(r) \, dr \approx \sum_{k=1}^N \max_{\tilde{r} \ge r_k} p(\tilde{r}) \cdot \Delta r_k$$

### Niveau 2: Mean Average Precision (mAP) over Alle Klassen
Vervolgens nemen we het rekenkundig gemiddelde (*Mean*) van deze AP-waarden over alle $C$ unieke klassen in de dataset:

$$\text{mAP} = \frac{1}{|C|} \sum_{c \in C} \text{AP}_c$$

### mAP@50 vs. mAP@50-95 (COCO Metric)
* **mAP@50:** Berekent de mAP bij een vaste IoU-drempel van $\text{IoU} \ge 0.50$. Dit is een vergevingsgezinde metric die enkel test of de box ruwweg op de juiste plek staat.
* **mAP@50-95:** De strenge COCO-standaard. Berekent de mAP op 10 opeenvolgende IoU-drempels ($0.50, 0.55, 0.60, \dots, 0.95$) en neemt daar weer het gemiddelde van. Dit bestraft slordige of te grote bounding boxes zwaar.

---
## 4. IoU (Intersection over Union) & NMS (Non-Maximum Suppression)

### 4.1 Intersection over Union (IoU)
$$\text{IoU}(A, B) = \frac{\text{Oppervlakte}(A \cap B)}{\text{Oppervlakte}(A \cup B)}$$

### 4.2 Non-Maximum Suppression (NMS) Algoritme
Een objectdetector genereert honderden overlappende ankerboxen rond één speler. NMS filtert deze tot één definitieve box:
1. Sorteer alle kandidaatboxen op hun betrouwbaarheidsscore ($conf$).
2. Kies de box $B_{max}$ met de hoogste score en voeg deze toe aan de definitieve selectie.
3. Bereken $\text{IoU}(B_{max}, B_i)$ voor alle overige overlappende boxen.
4. Verwijder alle boxen $B_i$ waarvan $\text{IoU} > \text{threshold}_{NMS}$.
5. Herhaal dit proces voor alle resterende boxen.

### Het Dilemma bij Kleine Objecten (Voetbal):
* Een te hoge $conf$-threshold ($0.25$) gooit subtiele baldetecties direct weg.
* Een te lage $conf$-threshold ($0.001$) veroorzaakt valse detecties op witte sokken en kalklijnen.
* Een te strikte NMS-threshold drukt de bal weg wanneer een speler er vlak naast staat.

---
## 5. Systematische Benchmark van 8 IoU & NMS Configuraties

Om de optimale balans te vinden tussen precisie, recall en kleine-baldetectie, hebben we een systematisch experiment uitgevoerd met 8 verschillende parametercombinaties op de validatiedataset.

In [ ]:
# Laad en Inspecteer de Benchmark Resultaten
import pandas as pd
from pathlib import Path

csv_path = None
for p in [
    Path("dimi/config/iou_nms_benchmark.csv"),
    Path("../config/iou_nms_benchmark.csv"),
    Path("config/iou_nms_benchmark.csv"),
    Path("dimi/iou_nms_benchmark.csv"),
    Path("../dimi/iou_nms_benchmark.csv")
]:
    if p.exists():
        csv_path = p
        break

if csv_path and csv_path.exists():
    df_bm = pd.read_csv(csv_path)
    print("=== IoU & NMS Benchmark Resultaten (8 Configuraties) ===")
    display_df = df_bm.copy()
    display_df.index = range(1, len(display_df) + 1)
    print(display_df.to_string())
else:
    print(f"Benchmark CSV niet gevonden.")

In [ ]:
# Vergelijkende Visualisatie van de 8 Configuraties
import matplotlib.pyplot as plt
import seaborn as sns

if 'df_bm' in locals() and not df_bm.empty:
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Barplot mAP@50 en mAP@50-95
    df_plot = df_bm.copy()
    df_plot['Short_Name'] = [
        'Ultra-High Rec', 'Tight NMS', 'WINNER (0.025/0.60)', 
        'Permissive NMS', 'Med Conf', 'Conservative', 
        'Default (0.25/0.60)', 'High Prec'
    ]
    
    x = range(len(df_plot))
    width = 0.35
    
    axes[0].bar([i - width/2 for i in x], df_plot['mAP@50'], width=width, label='mAP@50', color='#1f77b4', alpha=0.85)
    axes[0].bar([i + width/2 for i in x], df_plot['mAP@50-95'], width=width, label='mAP@50-95', color='#ff7f0e', alpha=0.85)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(df_plot['Short_Name'], rotation=40, ha='right', fontsize=10)
    axes[0].set_title('mAP@50 & mAP@50-95 Vergelijking per Configuratie', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Score')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Highlight de winnaar
    winner_idx = 2
    axes[0].get_children()[winner_idx].set_color('#00a86b')
    axes[0].get_children()[winner_idx + len(df_plot)].set_color('#2e8b57')
    
    # Scatterplot: Precision vs Recall Trade-off
    scatter = axes[1].scatter(
        df_plot['Recall (R)'], df_plot['Precision (P)'], 
        s=df_plot['mAP@50'] * 400, c=df_plot['F1-Score'], 
        cmap='viridis', edgecolors='black', linewidth=1.5, alpha=0.9
    )
    cbar = plt.colorbar(scatter, ax=axes[1])
    cbar.set_label('F1-Score')
    
    for i, row in df_plot.iterrows():
        axes[1].annotate(
            row['Short_Name'], (row['Recall (R)'] + 0.001, row['Precision (P)'] + 0.001),
            fontsize=9, fontweight='bold' if i == winner_idx else 'normal'
        )
        
    axes[1].set_title('Precision vs. Recall Trade-Off (Grootte = mAP@50)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Recall (R)')
    axes[1].set_ylabel('Precision (P)')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

---
## 6. De Winnende Configuratie & Verantwoording

### 🏆 De Winnaar: `Ball-Optimized (Balanced NMS)`
* **Parameters:** `conf = 0.025`, `iou = 0.60`
* **Prestaties:**
  * **mAP@50:** **0.7256** (Absoluut hoogste score van alle geteste configuraties)
  * **mAP@50-95:** **0.3086** (Hoogste box-localisatie score)
  * **Precisie (P):** **0.7382** | **Recall (R):** **0.6912** | **F1-Score:** **0.7139**
  * **Ball mAP@50:** **0.0215**

### Vergelijking met de YOLO Standaard Baseline (`conf=0.250, iou=0.60`):
* De standaard YOLO drempel behaalt slechts een mAP@50 van **0.6427**.
* Onze winnende configuratie levert een spectaculaire winst op van **+8.3% mAP@50** (en +13.0% op mAP@50-95).
* **Waarom?** 
  Een drempel van $0.25$ filtert kleine, deels bedekte voetballen rigoureus weg. Door de drempel te verlagen naar $0.025$ geven we de baldetecties de kans om naar de downstream tracking-pipeline door te stromen, terwijl NMS op $0.60$ voorkomt dat spelers dubbel worden gedetecteerd.

---
## 7. Kwalitatieve Error Analyse: Voetocclusies & Penaltystip Hallucinaties

Bij visuele inspectie van de detectieresultaten identificeren we twee fundamentele foutmodi:

### 7.1 False Negatives (Gemiste Bal bij Dribbels)
Wanneer een middenvelder dribbelt of een sliding maakt, bevindt de bal zich tussen twee krioelende benen. De contrastrijke textuur verdwijnt tijdelijk, waardoor de ruwe detectiescore zakt naar $0.01 - 0.02$.
* *Oplossing in de pipeline:* De downstream **InvariantBallTracker** activeert in dit geval de **Dribble Carrier Hold**, waarbij de positie van de bal automatisch wordt vastgeklikt aan de voeten van de actieve baldrager totdat de bal weer vrijkomt.

### 7.2 False Positives (De Penaltystip)
Op cameraframe positie $(1536, 378)$ bevindt zich de 11-meter penaltystip van het doelgebied. Omdat een penaltystip wit, rond en exact even groot is als een voetbal op afstand, detecteerde het model deze stip periodiek als bal.
* *Oplossing in de pipeline:* We hebben een **Micro-Penaltystip Filter** ontworpen in pitch-space op $(925, 340)$ met een straal van $20	ext{ px}$. Dit elimineert de valse penaltystip $100\%$ betrouwbaar, zonder echte passes door het strafschopgebied te onderdrukken.